# 02.1 — Tensors and Autograd

The two things everything in PyTorch is built on: **tensors**, which are NumPy arrays that can run on a GPU, and **autograd**, which computes gradients automatically.
It starts from NumPy (covered in the ML repo's `00-numpy-and-pandas`) and ends by training a model with nothing but tensors and autograd.

## What's in this notebook

- 1\. From NumPy arrays to tensors
- 2\. Creating tensors and random seeds
- 3\. Dtypes and memory
- 4\. Shapes: view, reshape, permute and contiguous memory
- 5\. Broadcasting and a silent shape bug
- 6\. Indexing, masks, gather and in-place operations
- 7\. Devices: CPU, CUDA and MPS
- 8\. Autograd: the computation graph
- 9\. Gradients add up: zero_grad and retain_graph
- 10\. no_grad, inference_mode and detach
- 11\. Non-scalar outputs and vector-Jacobian products
- 12\. Autograd vs the hand-written backprop from 01
- 13\. Custom autograd functions
- 14\. Training with raw tensors
- 15\. Common errors and what they mean
- 16\. Trade-offs
- 17\. Interview questions
- 18\. In my own words

In [1]:
import random
import time

import numpy as np
import torch
from torch import nn

print("PyTorch", torch.__version__, "| MPS available:", torch.backends.mps.is_available(), "| CUDA available:", torch.cuda.is_available())


def show_error(fn) -> None:
    """Run fn and print the first line of the error it raises, so the notebook can show real error messages."""
    try:
        fn()
        print("no error")
    except Exception as e:
        message = str(e).splitlines()[0]
        if len(message) > 220:                 # keep whole sentences
            message = message[: message.rfind(". ", 0, 220) + 1]
        print(f"{type(e).__name__}: {message}")

PyTorch 2.15.0 | MPS available: True | CUDA available: False


# 1. FROM NUMPY ARRAYS TO TENSORS

A **tensor** is an n-dimensional array, like a NumPy array, with two extra abilities:
it can live on a GPU, and it can record the operations done on it so that gradients can be computed automatically.
Most of the NumPy API carries over with small renames.

| NumPy | PyTorch | Note |
|---|---|---|
| `np.array(data)` | `torch.tensor(data)` | copies the data |
| `x.shape`, `x.dtype` | `x.shape` (or `x.size()`), `x.dtype` | |
| `axis=0` | `dim=0` | `axis=` is also accepted |
| `keepdims=True` | `keepdim=True` | |
| `x.reshape(...)` | `x.reshape(...)` or `x.view(...)` | section 4 |
| `np.concatenate`, `np.stack` | `torch.cat`, `torch.stack` | |
| `A @ B`, `np.matmul` | `A @ B`, `torch.matmul` | |
| `x.astype(np.float32)` | `x.to(torch.float32)` or `x.float()` | |
| `x.copy()` | `x.clone()` | |
| `np.random.seed(0)` | `torch.manual_seed(0)` | |

Moving between the two:
- `torch.tensor(array)` **copies** the data.
- `torch.from_numpy(array)` and `tensor.numpy()` **share** memory with the original (CPU tensors only): changing one changes the other.
  There is no copy, so it is fast, but it is also a source of surprising bugs.

In [2]:
a = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
t = torch.tensor(a)

print("sum over rows    NumPy:", a.sum(axis=0), "| PyTorch:", t.sum(dim=0))
print("mean, keep dims  NumPy:", a.mean(axis=1, keepdims=True).shape, "| PyTorch:", tuple(t.mean(dim=1, keepdim=True).shape))
print("same matrix product:", np.allclose(a @ a.T, (t @ t.T).numpy()))

copied, shared = torch.tensor(a), torch.from_numpy(a)
a[0, 0] = 100.0
print(f"\nafter a[0, 0] = 100:  torch.tensor copy -> {copied[0, 0].item()},  torch.from_numpy -> {shared[0, 0].item()}")

sum over rows    NumPy: [5. 7. 9.] | PyTorch: tensor([5., 7., 9.], dtype=torch.float64)
mean, keep dims  NumPy: (2, 1) | PyTorch: (2, 1)
same matrix product: True

after a[0, 0] = 100:  torch.tensor copy -> 1.0,  torch.from_numpy -> 100.0


The results are the same with `axis` renamed to `dim`. Note `dtype=torch.float64` in the PyTorch output: the tensor kept NumPy's default dtype (section 3). After the NumPy array changes, the `torch.tensor` copy still holds 1.0,
but the `from_numpy` tensor shows 100.0, because it is the same memory.

# 2. CREATING TENSORS AND RANDOM SEEDS

| Function | Gives |
|---|---|
| `torch.zeros(2, 3)`, `torch.ones(...)`, `torch.full((2, 3), 7.0)` | constant tensors |
| `torch.arange(0, 10, 2)`, `torch.linspace(0, 1, 5)` | ranges |
| `torch.rand(2, 3)` / `torch.randn(2, 3)` / `torch.randint(0, 10, (2, 3))` | uniform in [0, 1) / standard normal / random integers |
| `torch.zeros_like(x)`, `torch.randn_like(x)` | same shape, dtype and device as `x` |
| `torch.eye(3)` | identity matrix |

For reproducible runs, a training script seeds every random number generator once at the start: Python's `random`, NumPy and PyTorch.
`torch.manual_seed` covers the CPU and every GPU. Bit-for-bit identical results on a GPU also need `torch.use_deterministic_algorithms(True)`,
which makes some operations slower.

In [3]:
print(torch.arange(0, 10, 2), torch.linspace(0, 1, 5))
print(torch.full((2, 3), 7.0))


def set_seed(seed: int) -> None:
    """Seed every random number generator a training script uses."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)                # the CPU and every GPU


set_seed(42)
first = torch.randn(3)
set_seed(42)
second = torch.randn(3)
print("\nsame seed, same numbers:", torch.equal(first, second), first)

g = torch.Generator().manual_seed(0)       # an independent random stream, e.g. only for shuffling data
print("from its own generator:", torch.rand(3, generator=g))

tensor([0, 2, 4, 6, 8]) tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tensor([[7., 7., 7.],
        [7., 7., 7.]])

same seed, same numbers: True tensor([0.3367, 0.1288, 0.2345])
from its own generator: tensor([0.4963, 0.7682, 0.0885])


Seeding with 42 twice gives identical numbers. A separate `torch.Generator` keeps one random stream (for example, data shuffling)
independent of the others, so adding dropout to a model doesn't change the order the data arrives in.

# 3. DTYPES AND MEMORY

| dtype | Bytes | Range and precision | Used for |
|---|---|---|---|
| `float32` | 4 | max about 3.4e38, about 7 significant digits | the default for weights and inputs |
| `float64` | 8 | max about 1.8e308, about 16 digits | NumPy's default; rarely used in deep learning; not supported on Apple MPS |
| `float16` | 2 | max 65,504, about 3 digits | mixed-precision training and inference on older GPUs |
| `bfloat16` | 2 | the same range as float32, about 2 digits | mixed-precision training on modern GPUs and TPUs; most open LLMs are released in it |
| `int64` (long) | 8 | whole numbers | class labels and token ids |
| `int8`, 4-bit | 1, 0.5 | 256 / 16 distinct values | quantized LLM weights (folder 07) |
| `bool` | 1 | True / False | masks |

**The NumPy trap:** NumPy creates float64 by default, and `torch.tensor` and `torch.from_numpy` keep that dtype.
Layer weights are float32, so feeding a float64 tensor into a model fails. Convert with `.float()` when data comes from NumPy or pandas.

In [4]:
from_numpy = torch.from_numpy(np.random.rand(4, 2))
print("dtype from NumPy:", from_numpy.dtype, "| dtype from a list of Python floats:", torch.tensor([1.0, 2.0]).dtype)

layer = nn.Linear(2, 1)                    # float32 weights
show_error(lambda: layer(from_numpy))
print("after .float(): output shape", tuple(layer(from_numpy.float()).shape))

print()
for dtype in [torch.float32, torch.float16, torch.bfloat16, torch.int64]:
    x = torch.zeros(1_000_000, dtype=dtype)
    print(f"{str(dtype):<15} one million values = {x.element_size() * x.nelement() / 1e6:.0f} MB")

dtype from NumPy: torch.float64 | dtype from a list of Python floats: torch.float32
RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float
after .float(): output shape (4, 1)

torch.float32   one million values = 4 MB
torch.float16   one million values = 2 MB
torch.bfloat16  one million values = 2 MB
torch.int64     one million values = 8 MB


float16 and bfloat16 both use 2 bytes but split them differently: float16 spends more bits on precision, bfloat16 on range.

In [5]:
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    big = torch.tensor(70_000.0, dtype=dtype).item()
    near_one = torch.tensor(1.001, dtype=dtype).item()
    print(f"{str(dtype):<15} 70,000 -> {big:<9}  1.001 -> {near_one:.6f}   smallest step above 1: {torch.finfo(dtype).eps:.1e}")

torch.float32   70,000 -> 70000.0    1.001 -> 1.001000   smallest step above 1: 1.2e-07
torch.float16   70,000 -> inf        1.001 -> 1.000977   smallest step above 1: 9.8e-04
torch.bfloat16  70,000 -> 70144.0    1.001 -> 1.000000   smallest step above 1: 7.8e-03


float16 turns 70,000 into `inf`: anything above 65,504 overflows, which is why float16 training needs loss scaling (folder 03).
bfloat16 keeps the float32 range (70,000 becomes 70,144) but loses precision: 1.001 rounds to 1.0.
For training, range matters more than the last digits, so bfloat16 is the usual choice on hardware that supports it.

Memory is bytes per value times the number of values: a 7-billion-parameter LLM needs 28 GB in float32, 14 GB in bfloat16 and 3.5 GB at 4 bits.

# 4. SHAPES: VIEW, RESHAPE, PERMUTE AND CONTIGUOUS MEMORY

A tensor is a flat block of memory plus a **shape** and **strides**. The stride of a dimension is how many elements to skip
to move one step along it. Many shape operations only change the strides, without copying any data.

| Operation | What it does | Copies data? |
|---|---|---|
| `x.view(2, 3)` | a new shape over the same memory | never; fails if the memory layout doesn't allow it |
| `x.reshape(2, 3)` | the same as `view` when possible | only when it has to |
| `x.T`, `x.transpose(0, 1)`, `x.permute(2, 0, 1)` | reorders dimensions by changing the strides | no, but the result is no longer contiguous |
| `x.contiguous()` | lays the data out in order in new memory | yes, unless it is already contiguous |
| `x.unsqueeze(0)` / `x.squeeze(0)` | adds / removes a dimension of size 1 | no |
| `x.flatten(1)` | merges all dimensions from 1 on, e.g. images into vectors | only when it has to |

Typical uses: `unsqueeze(0)` turns one example into a batch of one, and `permute(0, 2, 3, 1)` converts images between
channels-first (PyTorch) and channels-last (NumPy, PIL, matplotlib).

In [6]:
x = torch.arange(6).reshape(2, 3)
print(x, "\nshape", tuple(x.shape), "| strides", x.stride(), "| contiguous", x.is_contiguous())

xt = x.T
print("\ntransposed: shape", tuple(xt.shape), "| strides", xt.stride(), "| contiguous", xt.is_contiguous(),
      "| same memory as x:", xt.data_ptr() == x.data_ptr())

show_error(lambda: xt.view(6))
print("reshape works:", xt.reshape(6), "| so does contiguous().view:", xt.contiguous().view(6))

image = torch.rand(3, 32, 32)              # channels, height, width
batch = image.unsqueeze(0)                 # a batch of one image
print("\nimage", tuple(image.shape), "-> batch", tuple(batch.shape),
      "-> channels last", tuple(batch.permute(0, 2, 3, 1).shape), "-> flattened", tuple(batch.flatten(1).shape))

tensor([[0, 1, 2],
        [3, 4, 5]]) 
shape (2, 3) | strides (3, 1) | contiguous True

transposed: shape (3, 2) | strides (1, 3) | contiguous False | same memory as x: True
RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape works: tensor([0, 3, 1, 4, 2, 5]) | so does contiguous().view: tensor([0, 3, 1, 4, 2, 5])

image (3, 32, 32) -> batch (1, 3, 32, 32) -> channels last (1, 32, 32, 3) -> flattened (1, 3072)


`x` has strides (3, 1): moving down one row skips 3 elements. The transpose only swaps them to (1, 3) and shares `x`'s memory,
so it costs nothing, but its elements are no longer stored in order. `view` refuses to work on it; `reshape` and `.contiguous()` make a copy.

# 5. BROADCASTING AND A SILENT SHAPE BUG

Broadcasting follows the same rules as NumPy (covered in the ML repo): shapes are compared from the right,
and a dimension of size 1 stretches to match. It is how a bias of shape (64,) gets added to every row of an (n, 64) batch.

The danger is that broadcasting never complains when shapes are *wrong but compatible*. The most common case:
model output of shape (n, 1) and targets of shape (n,). Subtracting them gives an (n, n) matrix, and the loss is silently wrong.

In [7]:
torch.manual_seed(0)
n = 5
pred = torch.randn(n, 1)                   # nn.Linear(..., 1) outputs shape (n, 1)
target = torch.randn(n)                    # a DataFrame column becomes shape (n,)

diff = pred - target
print("pred", tuple(pred.shape), "- target", tuple(target.shape), "=", tuple(diff.shape))
print(f"wrong MSE: {(diff ** 2).mean().item():.4f}   correct MSE: {((pred.squeeze(1) - target) ** 2).mean().item():.4f}")

pred (5, 1) - target (5,) = (5, 5)
wrong MSE: 2.2935   correct MSE: 4.0700


5 predictions and 5 targets produced 25 differences, and the loss came out as 2.2935 instead of 4.0700, with no error.
`nn.MSELoss` prints a warning for this case, but a hand-written loss or metric doesn't.
A cheap guard in training code: `assert pred.shape == target.shape` before computing the loss.

# 6. INDEXING, MASKS, GATHER AND IN-PLACE OPERATIONS

- Indexing, slicing and boolean masks work as in NumPy: `x[0]`, `x[:, 1:]`, `x[x > 0]`.
- `torch.where(cond, a, b)` takes values from `a` where `cond` is true and from `b` elsewhere.
- `torch.gather` picks one value per row by index. The classic use: given logits of shape (batch, classes) and the true labels,
  pick the score of the true class in each row. The same operation picks the log-probability of each generated token in an LLM,
  which is how DPO and RLHF losses are computed (folder 07).
- Operations ending in `_` work **in place**: `x.add_(1)`, `x.clamp_(0, 1)`, `x.zero_()`. They save memory but overwrite values
  that autograd may still need (section 15).

In [8]:
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 3.0, 0.2],
                       [1.0, 1.0, 4.0],
                       [0.3, 2.5, 0.0]])
labels = torch.tensor([0, 1, 2, 0])

print("values above 2:", logits[logits > 2])
print("first row with negatives set to 0:", torch.where(logits < 0, 0.0, logits)[0])

log_probs = torch.log_softmax(logits, dim=1)
true_class = log_probs.gather(1, labels.unsqueeze(1)).squeeze(1)
print("\nlog-probability of the true class:", true_class)
print("same with indexing, log_probs[arange, labels]:", torch.equal(true_class, log_probs[torch.arange(4), labels]))
print(f"cross-entropy by hand: {-true_class.mean().item():.4f}  |  nn.functional.cross_entropy: {nn.functional.cross_entropy(logits, labels).item():.4f}")

w = torch.ones(3)
w.mul_(2).add_(1)
print("\nin-place result:", w)

values above 2: tensor([3.0000, 4.0000, 2.5000])
first row with negatives set to 0: tensor([2.0000, 0.5000, 0.0000])

log-probability of the true class: tensor([-0.2413, -0.1096, -0.0949, -2.3764])
same with indexing, log_probs[arange, labels]: True
cross-entropy by hand: 0.7056  |  nn.functional.cross_entropy: 0.7056

in-place result: tensor([3., 3., 3.])


Cross-entropy is the mean of `−log p(true class)`, so `log_softmax` + `gather` reproduces `nn.functional.cross_entropy` exactly (0.7056 both ways).
The last row is labelled class 0 but the model favours class 1, so its log-probability (−2.376) is the most negative and dominates the loss.

# 7. DEVICES: CPU, CUDA AND MPS

| Device | Hardware | Notes |
|---|---|---|
| `cpu` | any machine | always available |
| `cuda` | NVIDIA GPUs | what training servers and cloud GPUs use; `cuda:0`, `cuda:1` with several GPUs |
| `mps` | Apple Silicon GPUs | good for local experiments; no float64, and a few operations are not implemented yet (`PYTORCH_ENABLE_MPS_FALLBACK=1` runs them on the CPU) |

The standard pattern: choose the device once, then move the model and every batch to it with `.to(device)`.
All tensors in one operation must be on the same device.

In [9]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = get_device()
print("device:", device)

x_gpu = torch.ones(3, device=device)
show_error(lambda: x_gpu + torch.ones(3))
show_error(lambda: torch.ones(3, dtype=torch.float64, device=device))

device: mps
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!
TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.


Two things make GPU timings misleading:
- **GPU operations are asynchronous:** Python queues the work and returns immediately. Stop the timer only after
  `torch.cuda.synchronize()` (or `torch.mps.synchronize()`). Calls such as `.item()`, `.cpu()` and `print(tensor)` also wait for the GPU.
- **Small operations don't benefit:** each GPU launch has a fixed overhead, and copying data between CPU and GPU memory takes time.
  GPUs win on large matrices and large batches.

In [10]:
def synchronize(device: torch.device) -> None:
    """Wait until all queued GPU work has finished (nothing to wait for on the CPU)."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def time_matmul(n: int, device: torch.device, repeats: int = 10) -> float:
    """Average milliseconds for one (n x n) @ (n x n) matrix multiply."""
    A, B = torch.randn(n, n, device=device), torch.randn(n, n, device=device)
    for _ in range(3):                     # warm-up: the first calls include one-off setup
        A @ B
    synchronize(device)
    start = time.perf_counter()
    for _ in range(repeats):
        A @ B
    synchronize(device)                    # wait until the GPU has really finished
    return (time.perf_counter() - start) / repeats * 1000


for n in [256, 1024, 4096]:
    cpu_ms, dev_ms = time_matmul(n, torch.device("cpu")), time_matmul(n, device)
    print(f"{n:>4} x {n:<4}  cpu {cpu_ms:7.2f} ms   {device.type} {dev_ms:7.2f} ms   speed-up {cpu_ms / dev_ms:.1f}x")

 256 x 256   cpu    0.04 ms   mps    0.12 ms   speed-up 0.3x


1024 x 1024  cpu    0.95 ms   mps    1.83 ms   speed-up 0.5x


4096 x 4096  cpu   65.21 ms   mps   34.14 ms   speed-up 1.9x


On this Mac the CPU wins on small matrices: 0.04 ms against 0.12 ms on MPS at 256 × 256, and 0.95 ms against 1.83 ms at 1024 × 1024, because every GPU launch has a fixed cost. At 4096 × 4096 the GPU is 1.9x faster (34 ms against 65 ms), and the gap keeps growing with size.
This Mac's CPU has fast matrix hardware built in, so the gap is small here. Against an NVIDIA data-centre GPU, large matrix multiplies are often 10x faster or more.

In [11]:
A = torch.randn(4096, 4096, device=device)
synchronize(device)
start = time.perf_counter()
C = A @ A
queued_ms = (time.perf_counter() - start) * 1000
synchronize(device)
finished_ms = (time.perf_counter() - start) * 1000
print(f"timer read right after the call: {queued_ms:.2f} ms   after synchronize: {finished_ms:.2f} ms")

X_cpu = torch.randn(4096, 4096)
start = time.perf_counter()
X_dev = X_cpu.to(device)
synchronize(device)
print(f"copying a 4096 x 4096 float32 tensor ({X_cpu.nelement() * 4 / 1e6:.0f} MB) to {device.type}: {(time.perf_counter() - start) * 1000:.2f} ms")

timer read right after the call: 0.24 ms   after synchronize: 50.02 ms


copying a 4096 x 4096 float32 tensor (67 MB) to mps: 3.39 ms


Read right after `A @ A`, the timer shows 0.24 ms, because the call only queued the work. After `synchronize` it shows 50 ms, the real cost. Timing GPU code without synchronizing reports a speed-up that doesn't exist.

Copying the 67 MB tensor to the GPU took 3.4 ms. Apple Silicon shares memory between the CPU and GPU, so copies are cheap here; on a server, data crosses the PCIe bus and copies cost more. That is why training code keeps the model on the GPU and copies each batch only once.

# 8. AUTOGRAD: THE COMPUTATION GRAPH

When a tensor has `requires_grad=True`, PyTorch records every operation on it in a **computation graph**.
`loss.backward()` walks that graph backwards with the chain rule and stores the gradient in each input's `.grad`.
This is the backpropagation written by hand in 01/1 section 7, done automatically.

- **Leaf tensors** are the ones you create, such as weights and inputs. `backward()` fills their `.grad`.
- Every result of an operation has a `grad_fn`: the backward function of the operation that created it.
- The graph is built **while the code runs** (define-by-run, or a dynamic graph). Plain Python `if` and `for` work inside a model,
  and each forward pass can take a different path. TensorFlow 1 built a fixed graph first and then ran it.
  `torch.compile` (folder 14) can turn the dynamic graph into optimized code when speed matters.

Example: `y = x² + 3x` at `x = 2`. By hand, `dy/dx = 2x + 3 = 7`.

In [12]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x
print("y =", y.item(), "| y.grad_fn:", type(y.grad_fn).__name__, "| is leaf: x", x.is_leaf, ", y", y.is_leaf)
print("the graph one step back from y:", [type(fn).__name__ for fn, _ in y.grad_fn.next_functions])

y.backward()
print("dy/dx =", x.grad.item())

y = 10.0 | y.grad_fn: AddBackward0 | is leaf: x True , y False
the graph one step back from y: ['PowBackward0', 'MulBackward0']
dy/dx = 7.0


Autograd gives 7.0, matching the hand calculation. `y` was made by an addition (`AddBackward0`) whose two inputs came from
a power and a multiplication; `backward()` runs those three backward functions in reverse order.

Because the graph is recorded as the code runs, ordinary Python branches just work:

In [13]:
def f(x):
    return x ** 3 if x > 0 else -x


for value in [2.0, -2.0]:
    x = torch.tensor(value, requires_grad=True)
    f(x).backward()
    print(f"f'({value}) = {x.grad.item()}   (path taken: {'x³, derivative 3x²' if value > 0 else '−x, derivative −1'})")

f'(2.0) = 12.0   (path taken: x³, derivative 3x²)
f'(-2.0) = -1.0   (path taken: −x, derivative −1)


# 9. GRADIENTS ADD UP: ZERO_GRAD AND RETAIN_GRAPH

- `backward()` **adds** to `.grad`; it doesn't overwrite it. That is why every training step calls `optimizer.zero_grad()`.
  It is deliberate: it makes **gradient accumulation** possible, i.e. summing the gradients of several small batches before one update,
  to train with a large effective batch that doesn't fit in memory (folder 03).
- `optimizer.zero_grad()` sets the gradients to `None` by default rather than filling them with zeros, which is slightly faster and saves memory.
- After `backward()`, PyTorch frees the values saved for the backward pass, so a second `backward()` through the same graph fails.
  `backward(retain_graph=True)` keeps them; it is rarely needed.

In [14]:
w = torch.tensor(3.0, requires_grad=True)
for step in range(3):
    loss = w ** 2                          # d(loss)/dw = 2w = 6
    loss.backward()
    print(f"after backward #{step + 1}: w.grad = {w.grad.item()}")

w.grad = None                              # what optimizer.zero_grad() does
(w ** 2).backward()
print("after clearing and one more backward:", w.grad.item())

loss = w ** 2
loss.backward()
show_error(lambda: loss.backward())

after backward #1: w.grad = 6.0
after backward #2: w.grad = 12.0
after backward #3: w.grad = 18.0
after clearing and one more backward: 6.0
RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad().


The true gradient is 6, but without clearing it grows to 18: three steps' worth added together.
Forgetting `zero_grad()` doesn't raise an error; training just behaves strangely, so it is a classic interview question.

# 10. NO_GRAD, INFERENCE_MODE AND DETACH

| Tool | What it does | Use it for |
|---|---|---|
| `with torch.no_grad():` | operations inside are not recorded | evaluation loops; updating weights by hand (section 14) |
| `with torch.inference_mode():` | like `no_grad`, and also skips some bookkeeping, so it is slightly faster; its results can never join a graph later | serving and batch prediction |
| `x.detach()` | the same data, cut off from the graph | stopping gradients through part of a model; logging; converting to NumPy |
| `p.requires_grad_(False)` | the parameter stops receiving gradients | freezing layers in transfer learning and fine-tuning |
| `loss.item()` | a plain Python number | logging the loss |

A classic memory leak: `losses.append(loss)` keeps every loss tensor *and its whole graph* alive. Use `losses.append(loss.item())`.

In [15]:
w = torch.randn(1000, 1000, requires_grad=True)
X = torch.randn(64, 1000)

out = X @ w
print("normal:          requires_grad =", out.requires_grad, "| grad_fn =", type(out.grad_fn).__name__)
with torch.no_grad():
    out = X @ w
print("no_grad:         requires_grad =", out.requires_grad, "| grad_fn =", out.grad_fn)
with torch.inference_mode():
    out = X @ w
print("inference_mode:  requires_grad =", out.requires_grad, "| inference tensor =", out.is_inference())

y = X @ w
d = y.detach()
print("detach():        requires_grad =", d.requires_grad, "| same memory as y:", d.data_ptr() == y.data_ptr())

normal:          requires_grad = True | grad_fn = MmBackward0
no_grad:         requires_grad = False | grad_fn = None
inference_mode:  requires_grad = False | inference tensor = True
detach():        requires_grad = False | same memory as y: True


Inside `no_grad` and `inference_mode` nothing is recorded, so no memory is kept for a backward pass.
`detach()` returns the same memory without the graph, so it is free, but writing into the detached tensor also changes `y`.

# 11. NON-SCALAR OUTPUTS AND VECTOR-JACOBIAN PRODUCTS

`backward()` with no arguments only works on a single number, such as a loss. For a vector output `y`, the full derivative is a
**Jacobian** matrix (one row per output, one column per input), which can be enormous. Autograd never builds it. Instead it computes
a **vector-Jacobian product** `vᵀ·J` for a vector `v` you pass in: `y.backward(v)`. For a scalar loss, `v` is simply 1.

This is **reverse-mode** differentiation: one backward pass gives the gradient of one output with respect to *all* inputs. A neural network has
millions of inputs (the weights) and one output (the loss), so reverse mode fits exactly. Forward mode would need one pass per weight.

`torch.autograd.grad(outputs, inputs)` returns the gradients instead of writing them to `.grad`. It is used for gradient penalties and second derivatives.

In [16]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2                                 # a vector output; dy_i/dx_i = 2·x_i
show_error(lambda: y.backward())

y.backward(torch.ones(3))                  # v = all ones, the same as y.sum().backward()
print("gradient with v = ones:", x.grad)
print("full Jacobian:\n", torch.autograd.functional.jacobian(lambda t: t ** 2, x.detach()))

x = torch.tensor(3.0, requires_grad=True)
(first,) = torch.autograd.grad(x ** 3, x, create_graph=True)   # 3x² = 27, keeping its own graph
(second,) = torch.autograd.grad(first, x)                      # 6x = 18
print("\nfirst derivative of x³ at 3:", first.item(), "| second derivative:", second.item())

RuntimeError: grad can be implicitly created only for scalar outputs


gradient with v = ones: tensor([2., 4., 6.])
full Jacobian:
 tensor([[2., 0., 0.],
        [0., 4., 0.],
        [0., 0., 6.]])

first derivative of x³ at 3: 27.0 | second derivative: 18.0


The Jacobian of an element-wise square is diagonal, and `vᵀ·J` with `v` = ones gives its diagonal, [2, 4, 6], without building the matrix.
`create_graph=True` records the backward pass itself, so it can be differentiated again: 18 for the second derivative of x³ at 3.

# 12. AUTOGRAD VS THE HAND-WRITTEN BACKPROP FROM 01

The same 2 → 64 → 3 network as in 01/1, written with raw tensors, and its gradients computed two ways:
by autograd, and by the formulas from 01/1 section 7. It uses float64 on the CPU so that rounding can't hide a real difference.

In [17]:
torch.manual_seed(0)
n = 32
X = torch.randn(n, 2, dtype=torch.float64)
y = torch.randint(0, 3, (n,))
W1 = (torch.randn(2, 64, dtype=torch.float64) * (2 / 2) ** 0.5).requires_grad_()     # He initialization
b1 = torch.zeros(64, dtype=torch.float64, requires_grad=True)
W2 = (torch.randn(64, 3, dtype=torch.float64) * (2 / 64) ** 0.5).requires_grad_()
b2 = torch.zeros(3, dtype=torch.float64, requires_grad=True)

Z1 = X @ W1 + b1
A1 = torch.relu(Z1)
Z2 = A1 @ W2 + b2
loss = nn.functional.cross_entropy(Z2, y)
loss.backward()                            # autograd

with torch.no_grad():                      # by hand, the formulas from 01/1
    dZ2 = (torch.softmax(Z2, dim=1) - nn.functional.one_hot(y, 3)) / n
    dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
    by_hand = {"W1": X.T @ dZ1, "b1": dZ1.sum(0), "W2": A1.T @ dZ2, "b2": dZ2.sum(0)}

for name, param in {"W1": W1, "b1": b1, "W2": W2, "b2": b2}.items():
    print(f"{name}: shape {str(tuple(param.grad.shape)):<8}  max difference autograd vs by hand = {(param.grad - by_hand[name]).abs().max().item():.1e}")

W1: shape (2, 64)   max difference autograd vs by hand = 1.4e-17
b1: shape (64,)     max difference autograd vs by hand = 6.9e-18
W2: shape (64, 3)   max difference autograd vs by hand = 5.6e-17
b2: shape (3,)      max difference autograd vs by hand = 1.4e-17


The largest difference is 5.6e-17, which is float64 rounding: autograd computes exactly what the hand-written backward pass in 01/1 computed,
without anyone writing it.

`torch.autograd.gradcheck` does the gradient check from 01/1 section 8 for any function: it compares autograd with numerical differences.

In [18]:
def two_layer_loss(W1, W2):
    return nn.functional.cross_entropy(torch.relu(X @ W1) @ W2, y)


print("gradcheck passed:", torch.autograd.gradcheck(two_layer_loss, (W1.detach().requires_grad_(), W2.detach().requires_grad_())))

gradcheck passed: True


# 13. CUSTOM AUTOGRAD FUNCTIONS

Sometimes you write the backward pass yourself: for an operation PyTorch doesn't have, for a faster or more stable gradient,
or for a deliberately "wrong" gradient. Subclass `torch.autograd.Function` with a `forward` and a `backward`;
`ctx.save_for_backward` keeps whatever the backward pass needs, exactly as the forward pass in 01/1 saved Z1 and A1.

In [19]:
class Sigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        out = 1 / (1 + torch.exp(-x))
        ctx.save_for_backward(out)         # the derivative only needs the output
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (out,) = ctx.saved_tensors
        return grad_output * out * (1 - out)


x = torch.randn(5, dtype=torch.float64, requires_grad=True)
print("custom sigmoid matches torch.sigmoid:", torch.allclose(Sigmoid.apply(x), torch.sigmoid(x)))
print("gradcheck passed:", torch.autograd.gradcheck(Sigmoid.apply, (x,)))

custom sigmoid matches torch.sigmoid: True
gradcheck passed: True


The best-known "wrong gradient" is the **straight-through estimator (STE)**, used in quantization-aware training.
Rounding has a gradient of 0 almost everywhere, so nothing before it could learn. The STE rounds in the forward pass
and passes the gradient through unchanged in the backward pass.

In [20]:
class RoundSTE(torch.autograd.Function):
    """Round in the forward pass; pass the gradient straight through in the backward pass."""

    @staticmethod
    def forward(ctx, x):
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


x = torch.tensor([0.4, 1.6, 2.5], requires_grad=True)
torch.round(x).sum().backward()
print("torch.round: values", torch.round(x).tolist(), "| gradient", x.grad.tolist())

x.grad = None
RoundSTE.apply(x).sum().backward()
print("RoundSTE:    values", RoundSTE.apply(x).tolist(), "| gradient", x.grad.tolist())

torch.round: values [0.0, 2.0, 2.0] | gradient [0.0, 0.0, 0.0]
RoundSTE:    values [0.0, 2.0, 2.0] | gradient [1.0, 1.0, 1.0]


Both round to the same values. With `torch.round` the gradient is [0, 0, 0], so a model with rounding inside could not train;
the STE passes [1, 1, 1]. Note that 2.5 rounds to 2.0: PyTorch rounds halves to the nearest even number, like NumPy.

# 14. TRAINING WITH RAW TENSORS

Everything so far, put together: linear regression trained with tensors and autograd only, with no `nn.Module` and no optimizer.
The data comes from `y = 3·x1 − 2·x2 + 1` plus noise, so we know what training should find.
This loop is what `nn.Linear`, `torch.optim.SGD` and `optimizer.zero_grad()` do for you, the subject of the next notebook (02.2).

In [21]:
torch.manual_seed(0)
X = torch.randn(1000, 2)
y = X @ torch.tensor([3.0, -2.0]) + 1.0 + 0.1 * torch.randn(1000)

w = torch.zeros(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for step in range(101):
    pred = X @ w + b
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():                  # the update itself must not be recorded
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad = None                          # clear for the next step
    b.grad = None
    if step % 25 == 0:
        print(f"step {step:>3}: loss {loss.item():.4f}")

print(f"\nlearned w = {w.detach().numpy().round(3)}, b = {b.item():.3f}   (true: w = [3, -2], b = 1)")

step   0: loss 13.9975
step  25: loss 0.0095
step  50: loss 0.0093
step  75: loss 0.0093
step 100: loss 0.0093

learned w = [ 2.999 -1.997], b = 0.997   (true: w = [3, -2], b = 1)


The loss falls from 14.00 to 0.0093, which is about the noise variance (0.1² = 0.01), and the learned weights are [2.999, −1.997] with b = 0.997:
autograd found the true line. Swapping the hand-written update for `torch.optim.SGD` and the raw `w`, `b` for `nn.Linear(2, 1)` gives the standard training loop.

# 15. COMMON ERRORS AND WHAT THEY MEAN

Each of these shows up in real projects, and the first line of the error usually points straight at the fix.

In [22]:
def update_without_no_grad():
    w = torch.zeros(2, requires_grad=True)
    (w.sum() * 2).backward()
    w -= 0.1 * w.grad


def modify_saved_value():
    a = torch.ones(3, requires_grad=True)
    s = a.sigmoid()                        # sigmoid saves its output for the backward pass...
    s.mul_(2)                              # ...and this in-place multiply overwrites it
    s.sum().backward()


cases = {
    "float labels for cross_entropy": lambda: nn.functional.cross_entropy(torch.randn(4, 3), torch.tensor([0.0, 1.0, 2.0, 1.0])),
    "NumPy float64 into a float32 layer": lambda: nn.Linear(2, 1)(torch.from_numpy(np.random.rand(3, 2))),
    "tensors on two devices": lambda: torch.ones(3, device=device) + torch.ones(3),
    "layer input size doesn't match the data": lambda: nn.Linear(4, 2)(torch.randn(8, 3)),
    ".numpy() on a tensor that needs gradients": lambda: torch.ones(2, requires_grad=True).numpy(),
    "updating weights outside no_grad": update_without_no_grad,
    "in-place change of a saved value": modify_saved_value,
}
for name, fn in cases.items():
    print(f"{name}:\n    ", end="")
    show_error(fn)

float labels for cross_entropy:
    RuntimeError: expected target dtype to be Long or Byte, but got Float
NumPy float64 into a float32 layer:
    RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float
tensors on two devices:
    RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!
layer input size doesn't match the data:
    RuntimeError: addmm: mat1 and mat2 shapes cannot be multiplied (8x3 and 4x2)
.numpy() on a tensor that needs gradients:
    RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.
updating weights outside no_grad:
    RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.
in-place change of a saved value:
    RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [3]], which is output 0 of Sigmoid, is at version 1; expected version 0 inst

| Error contains | Cause | Fix |
|---|---|---|
| `expected target dtype to be Long` | labels are float or int32 | `labels.long()` |
| `mat1 and mat2 must have the same dtype` | float64 data from NumPy or pandas | `.float()` when building the dataset |
| `Expected all tensors to be on the same device` | model on the GPU, batch on the CPU, or the reverse | `.to(device)` for the model and for every batch |
| `mat1 and mat2 shapes cannot be multiplied` | the layer's `in_features` doesn't match the data | print the shapes; fix `in_features` |
| `Can't call numpy() on Tensor that requires grad` | converting a tensor that is still in the graph | `.detach().cpu().numpy()` |
| `a leaf Variable that requires grad is being used in an in-place operation` | a manual weight update that autograd would record | wrap the update in `torch.no_grad()` |
| `one of the variables needed for gradient computation has been modified by an inplace operation` | an in-place operation overwrote a value saved for backward | use the out-of-place version, `x = x * 2` |
| `Trying to backward through the graph a second time` | `backward()` called twice on one graph | build a new graph every step |

# 16. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Training precision | float32 / bfloat16 mixed / float16 mixed | float32: simple and safe, but twice the memory and slower. bfloat16: half the memory and much faster on modern GPUs, with the float32 range but fewer digits. float16: same savings, but overflows above 65,504, so it needs loss scaling | bfloat16 mixed precision where the GPU supports it, float32 otherwise (folder 03) |
| `view` vs `reshape` | view / reshape | view: guaranteed no copy, but fails on non-contiguous tensors. reshape: always works, but may copy without telling you | `reshape`, unless shared memory matters |
| NumPy to tensor | `torch.tensor` / `torch.from_numpy` | `torch.tensor`: an independent copy, safe. `from_numpy`: no copy and fast, but changes to either side show up in the other | `from_numpy` for large arrays nobody modifies afterwards |
| In-place operations | `x.add_()` / `x = x + ...` | in-place: saves memory, but can break autograd (section 15) and makes code harder to follow. Out-of-place: always safe | out-of-place; in-place only inside `no_grad` or for large buffers |
| Turning off gradients | `no_grad` / `inference_mode` | no_grad: results can still be used with autograd later. inference_mode: a little faster, but its results can never join a graph | `inference_mode` for serving, `no_grad` for evaluation during training |
| Device | CPU / MPS / CUDA | CPU: no setup, slow for large models. MPS: free on a Mac, but no float64 and some missing operations. CUDA: the industry standard, costs money | develop on CPU or MPS with small data; train on CUDA |
| Gradients | autograd / custom `Function` | autograd: correct and no extra code. custom Function: control over the gradient (STE) or memory, but you own its correctness, so test it with `gradcheck` | autograd, unless you need a special gradient |
| Execution | eager / `torch.compile` | eager: easy to debug line by line. compile: faster training and serving, but adds compile time and is harder to debug | eager while developing; compile once the model works (folder 14) |

# 17. INTERVIEW QUESTIONS

### Q1. What is a tensor, and how does PyTorch's autograd work, in 30 seconds?
A tensor is an n-dimensional array, like a NumPy array, that can also run on a GPU and record the operations applied to it.
When a tensor has `requires_grad=True`, PyTorch builds a computation graph during the forward pass. `loss.backward()` walks that graph in reverse,
applies the chain rule at each operation, and adds the gradient of the loss to each leaf tensor's `.grad`. The optimizer then uses those gradients to update the weights.

### Q2. How is a PyTorch tensor different from a NumPy array?
Three things: it can live on a GPU (`.to("cuda")`), it can track gradients, and its default float type is float32 instead of float64.
The APIs are close (`dim` instead of `axis`). `torch.from_numpy` and `.numpy()` share memory on the CPU without copying; `torch.tensor` copies.

### Q3. What is a computation graph? Dynamic vs static graphs?
A graph whose nodes are operations and whose edges are tensors, recorded so the backward pass can apply the chain rule.
PyTorch builds it while the code runs (dynamic, define-by-run): Python control flow works and debugging is ordinary Python debugging.
Static graphs (TensorFlow 1) are defined first and then run, which allows more optimization. `torch.compile` gives PyTorch much of that speed while keeping the eager style.

### Q4. Why do we call `optimizer.zero_grad()`? Why does PyTorch add gradients up instead of replacing them?
`backward()` adds to `.grad`, so without clearing, each step would use the sum of all previous gradients. It adds up on purpose,
to support gradient accumulation: several small batches' gradients are summed before one optimizer step, simulating a larger batch.
It also lets one parameter receive gradient from several parts of the graph.

### Q5. `no_grad` vs `inference_mode` vs `detach()` vs `requires_grad=False`?
`no_grad` and `inference_mode` are contexts in which nothing is recorded; `inference_mode` is slightly faster, but its outputs can never be used with autograd later.
`detach()` cuts one tensor out of the graph (same memory). `requires_grad_(False)` on parameters freezes them, as in transfer learning or fine-tuning only some layers.

### Q6. `view` vs `reshape`? What does contiguous mean?
A tensor is memory plus a shape and strides. Contiguous means its elements are stored in the order its shape implies.
`view` only changes the shape metadata and fails if the memory layout doesn't allow it (for example after a transpose); `reshape` falls back to copying.
`transpose` and `permute` just swap strides, so they are free but make the tensor non-contiguous.

### Q7. float32 vs float16 vs bfloat16? Why do LLMs use bfloat16?
float16 and bfloat16 both use 2 bytes, half of float32. float16 has more precision but a maximum of 65,504, so values overflow and training needs loss scaling.
bfloat16 has the same exponent range as float32 with fewer digits, so it rarely overflows and needs no loss scaling. Training cares more about range than precision,
so bfloat16 mixed precision is the default on modern GPUs.

### Q8. How much memory does a 7B-parameter model need?
Weights alone: 7B × bytes per parameter, so 28 GB in float32, 14 GB in bfloat16, 7 GB in int8 and about 3.5 GB in 4-bit.
Full training needs much more: with Adam in mixed precision, about 16 bytes per parameter for weights, gradients and optimizer state (about 112 GB), plus activations.
That is why fine-tuning large models uses LoRA or QLoRA (folder 13).

### Q9. Why must GPU code be timed with `synchronize()`? Why can the GPU be slower than the CPU?
GPU operations are asynchronous: the Python call returns as soon as the work is queued, so a timer without synchronization measures almost nothing.
For small operations, the fixed cost of launching GPU work and copying data outweighs the faster maths, so the GPU only wins on large matrices and batches.

### Q10. What is reverse-mode automatic differentiation, and why do neural networks use it?
Reverse mode computes vector-Jacobian products by going backwards from the output. One backward pass gives the gradient of one scalar output with respect to all inputs.
Training has one scalar (the loss) and millions of inputs (the weights), so reverse mode costs about one extra forward pass, whereas forward mode would need one pass per weight.

### Q11. What causes "one of the variables needed for gradient computation has been modified by an inplace operation"?
An in-place operation (`x.mul_()`, `x += 1`, `relu(inplace=True)`) overwrote a tensor that autograd had saved for the backward pass.
PyTorch tracks a version counter on each tensor and detects the change. The fix is the out-of-place version of the operation.

### Q12. What does `torch.gather` do? Give a use.
It picks values along a dimension using an index tensor, for example one value per row. In classification it picks the log-probability of the true class
in each row, which is cross-entropy. In LLM training it picks the log-probability of each actual next token, which DPO and RLHF losses are built from.

### Q13. Give an example of a silent broadcasting bug.
Predictions of shape (n, 1) minus targets of shape (n,) broadcast to an (n, n) matrix, so an MSE computed by hand is wrong with no error.
Guard with `assert pred.shape == target.shape`, or squeeze the prediction.

### Q14. When would you write a custom `torch.autograd.Function`?
When an operation needs a gradient autograd can't produce or produces badly: the straight-through estimator for rounding in quantization-aware training,
a numerically stabler or memory-saving backward pass, or a custom GPU kernel. Verify it with `torch.autograd.gradcheck` in float64.

### Q15. How do you make a PyTorch training run reproducible?
Seed Python's `random`, NumPy and PyTorch at the start, give the DataLoader a seeded generator, and on GPUs set `torch.use_deterministic_algorithms(True)`.
Also log library versions and the config. Even then, results can differ across GPU models and library versions.

# 18. IN MY OWN WORDS

Write this from memory, without looking above:

- What a tensor adds on top of a NumPy array:
- What happens, step by step, when `loss.backward()` runs:
- Why gradients must be cleared every step, and what accumulation is for:
- When to use `no_grad`, `inference_mode` and `detach()`:
- float32 vs bfloat16, and the memory a 7B model needs: